Create Pass Fail Labels

In [17]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [18]:
import json
from pathlib import Path
import pandas as pd

In [19]:
annotation_path = Path("../data/raw/train/_annotations.coco.json")

with open(annotation_path, "r") as file:
    coco_data = json.load(file)
annotated_image_ids = set()

for annotation in coco_data["annotations"]:
    annotated_image_ids.add(annotation["image_id"])

In [20]:
# Create a lookup from category ID to category name
category_names = {}

for category in coco_data["categories"]:
    category_names[category["id"]] = category["name"]


# Count annotations per image
annotation_count = {}

for annotation in coco_data["annotations"]:
    image_id = annotation["image_id"]

    if image_id not in annotation_count:
        annotation_count[image_id] = 0

    annotation_count[image_id] += 1


# Store defect types for each image
defect_types = {}

for annotation in coco_data["annotations"]:
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    defect_name = category_names[category_id]

    if image_id not in defect_types:
        defect_types[image_id] = []

    defect_types[image_id].append(defect_name)


# Build PASS / FAIL label table
image_labels = []

for image_record in coco_data["images"]:

    image_id = image_record["id"]
    filename = image_record["file_name"]

    defects = defect_types.get(image_id, [])
    number_of_defects = len(defects)

    if number_of_defects > 0:
        label = "FAIL"
    else:
        label = "PASS"

    image_labels.append({
        "image_id": image_id,
        "filename": filename,
        "number_of_defects": number_of_defects,
        "defect_types": defects,
        "label": label
    })

labels_df = pd.DataFrame(image_labels)

labels_df.head()

,image_id,filename,number_of_defects,defect_types,label
0,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,1,[packaging-error],FAIL
1,1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,1,[unsealed],FAIL
2,2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,1,[unsealed],FAIL
3,3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,0,[],PASS
4,4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,1,[wrinkle],FAIL


In [21]:
labels_df = pd.DataFrame(image_labels)

labels_df.head()

,image_id,filename,number_of_defects,defect_types,label
0,0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,1,[packaging-error],FAIL
1,1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,1,[unsealed],FAIL
2,2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,1,[unsealed],FAIL
3,3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,0,[],PASS
4,4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,1,[wrinkle],FAIL


In [22]:
print(labels_df["label"].value_counts())
print(labels_df["label"].value_counts(normalize=True))

label
FAIL    2665
PASS     684
Name: count, dtype: int64
label
FAIL    0.79576
PASS    0.20424
Name: proportion, dtype: float64


In [23]:
labels_df["label"].value_counts()

label
FAIL    2665
PASS     684
Name: count, dtype: int64